### Links and Resources
| Function | Use |
|---|---|
| [`round`, `bround`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.round.html) | HALF_UP vs banker's rounding |
| [`ceil`, `floor`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.ceil.html) | Round toward ±∞ |
| [`abs`, `sign`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.abs.html) | Magnitude / direction |
| [`pow`, `sqrt`, `log`, `exp`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pow.html) | Exponents and logs |
| [`rand`, `randn`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.rand.html) | Uniform / normal random |
| [`greatest`, `least`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.greatest.html) | Row-wise max/min across columns |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# Numeric and Math Functions

A small but well-used corner of the API. Most are 1:1 with their `math` module counterparts, but there are two traps worth flagging up front:

- **`round` is HALF_UP, `bround` is banker's rounding.** Financial code usually wants `bround` to avoid statistical bias.
- **`rand()` is non-deterministic by default**, which breaks job re-runs. Always pass a seed.

In [ ]:
df = spark.createDataFrame([
    (1, 12.345,  -7.0, 0),
    (2,  0.5,     2.5, 3),
    (3, -3.7,     1.0, 5),
    (4, 99.999,  -0.0, 2),
], ["id", "x", "y", "n"])
display(df)

## Rounding — `round`, `bround`, `ceil`, `floor`

`round(0.5) = 1` (HALF_UP). `bround(0.5) = 0`, `bround(2.5) = 2` (round half to even). The difference matters when you sum many rounded values.

In [ ]:
df.select(
    "x",
    F.round("x", 1).alias("round_1dp"),
    F.bround("x", 1).alias("bround_1dp"),
    F.ceil("x").alias("ceil"),
    F.floor("x").alias("floor"),
).show()

## Magnitude and sign — `abs`, `signum`

In [ ]:
df.select("y", F.abs("y").alias("abs"), F.signum("y").alias("sign")).show()

## Exponents and roots — `pow`, `sqrt`, `cbrt`, `exp`, `log`, `log10`, `log2`

`log(b, x)` is `log_b(x)`. Single-argument `log(x)` is natural log.

In [ ]:
df.select(
    "x",
    F.pow("x", F.lit(2)).alias("squared"),
    F.sqrt(F.abs("x")).alias("sqrt_abs"),
    F.log(F.lit(10.0), F.abs("x") + F.lit(1)).alias("log10_x_plus_1"),
).show()

## Modulo and division — `%`, `pmod`

`%` follows the sign of the dividend. `pmod` is always non-negative — what you usually want for hashing/bucketing.

In [ ]:
df.select(
    "y",
    (F.col("y") % 3).alias("mod_3"),
    F.pmod("y", F.lit(3)).alias("pmod_3"),
).show()

## Row-wise extremes — `greatest`, `least`

These take any number of columns and return the max/min **across them, per row** — different from the `max()`/`min()` aggregates.

In [ ]:
df.select(
    "x", "y", "n",
    F.greatest("x", "y", "n").alias("greatest"),
    F.least("x", "y", "n").alias("least"),
).show()

## Random numbers — always seed!

`rand()` and `randn()` (Gaussian) without a seed change every run, breaking idempotence. Pass an integer seed.

In [ ]:
spark.range(4).select(
    "id",
    F.rand(seed=42).alias("uniform"),
    F.randn(seed=42).alias("normal"),
).show()

## Casting numeric types

Use `.cast("int" / "long" / "double" / "decimal(18,2)")`. Casting a string that isn't a number returns NULL — silent — so guard with `try_cast` (Spark 3.4+) when reading dirty input.

In [ ]:
spark.createDataFrame([("12.5",), ("abc",), ("3",)], ["s"]).select(
    "s",
    F.col("s").cast("double").alias("cast_double"),
    F.expr("try_cast(s AS DOUBLE)").alias("try_cast_double"),
).show()

## Takeaways

- For money: `bround` over `round`, or use `decimal(p, s)` end-to-end.
- `pmod` for hashing; `%` only when sign of dividend is meaningful.
- `greatest`/`least` are row-wise — the agg counterparts (`max`/`min`) are different functions.
- Always pass a seed to `rand()` / `randn()`.
- `try_cast` returns NULL on bad input instead of failing the job.